# Semestrální projekt (Část 1): Exploratory Data Analysis & Clustering

**Student:** Ataniyaz Aman (AMA0023) 
**Dataset:** Melbourne Housing Snapshot  
**Předmět:** Základy strojového učení

---

## 1. Úvod a cíle projektu

Cílem této části projektu je provést průzkumnou analýzu dat (EDA) a pomocí shlukování (clusteringu) rozdělit nemovitosti v Melbourne do skupin podle jejich vlastností a vzdálenosti od centra.

Práce zahrnuje následující kroky:

1. **Dataset Validation:** Ověření velikosti, datových typů, chybějících hodnot a duplicit.
2. **Data Cleaning & Feature Engineering:** Příprava dat a vytvoření tří nových příznaků.
3. **Deep Exploratory Data Analysis:** Analýza rozdělení, vztahů a korelací s interpretací grafů.
4. **Cluster Analysis:** Porovnání K-Means a hierarchického shlukování, výběr počtu skupin a jejich popis.

**Zdroj dat:** [Melbourne Housing Snapshot — Dan Becker, Kaggle](https://www.kaggle.com/datasets/dansbecker/melbourne-housing-snapshot). Jedna řádka představuje záznam o prodeji; ceny jsou v AUD, plochy v m² a vzdálenost v km.

**Zaměření analýzy:** Cena nebude vstupem clusteringu. Použijeme ji až k následnému porovnání nalezených skupin.

### 1.1. Import knihoven a nastavení prostředí


In [ ]:
# Základní nástroje a knihovny pro práci s daty
from pathlib import Path
import platform
import numpy as np
import pandas as pd

# Vizualizace
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

# Předzpracování, shlukování a vyhodnocení
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, cut_tree, dendrogram

# Reprodukovatelnost a jednotné nastavení grafů
SEED = 42
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 20)
print('Python:', platform.python_version())
print('pandas:', pd.__version__, '| scikit-learn:', sklearn.__version__)


### 1.2. Načtení a základní kontrola dat (Dataset Validation)
Původní tabulku `df_raw` zachováme. Úpravy budeme dělat v samostatné kopii.
`shape` ukazuje počet řádků a sloupců; samo o sobě ale neověřuje počet použitelných příznaků.


In [ ]:
# 1. Načtení souboru
DATA_PATH = Path('/Users/aatanik/Documents/Fundamentals-of-Machine-Learning-ZSU-/Semester_Project/melb_data.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Uložte melb_data.csv do stejné pracovní složky jako notebook.')
df_raw = pd.read_csv(DATA_PATH)

# 2. Kontrola rozměrů, duplicit a datových typů
print('Rozměry:', df_raw.shape)
print('Plně duplicitní řádky:', df_raw.duplicated().sum())
display(df_raw.head())
df_raw.info()


### 1.3. Datové typy a splnění požadavků

| Skupina | Příklady | Poznámka |
|---|---|---|
| Spojité veličiny | `Distance`, `Landsize`, `BuildingArea`, `Price` | Vzdálenost, plochy a cena; hodnoty mohou být zaokrouhlené. |
| Počty | `Rooms`, `Bedroom2`, `Bathroom`, `Car`, `Propertycount` | Diskrétní číselné proměnné, nikoli spojité. |
| Kategorie | `Type`, `Suburb`, `Method`, `SellerG`, `CouncilArea`, `Regionname`, `Postcode` | Číselně uložené PSČ není spojitá veličina. |
| Čas | `Date`, `YearBuilt` | Datum prodeje a rok výstavby. |
| Poloha | `Lattitude`, `Longtitude` | Souřadnice; ponechány s původním názvem. |
| Identifikační text | `Address` | Adresa se nehodí jako přímý vstup vzdálenostního modelu. |

Dataset má **13 580 řádků a 21 sloupců**. I po konzervativním vynechání ceny,
adresy a podobného sloupce `Bedroom2` zbývá 18 sloupců, tedy více než požadovaných deset.
Minimálně tři spojité veličiny máme i bez ceny: vzdálenost a dvě plochy.
Typ nemovitosti a způsob prodeje jsou dva příklady kategorií. Data obsahují chybějící hodnoty a datum jako text.
Požadavky na dataset jsou tak splněny. Pro clustering později vybereme užší, zdůvodněnou sadu.

### 1.4. Analýza chybějících hodnot (Missing Values)

Zjistíme počet a podíl chybějících hodnot. Výsledek použijeme při volbě strategie doplnění dat.


In [ ]:
# 1. Počet a podíl chybějících hodnot
missing = pd.DataFrame({
    'počet': df_raw.isna().sum(),
    'podíl (%)': df_raw.isna().mean().mul(100).round(2)
}).query('počet > 0').sort_values('počet', ascending=False)
display(missing)

# 2. Vizualizace rozsahu chybějících dat
fig, ax = plt.subplots(figsize=(8, 3.5))
missing['podíl (%)'].sort_values().plot.barh(ax=ax, color='#337d9e')
ax.set(xlabel='Chybějící hodnoty (%)', ylabel='', title='Chybějící hodnoty v původních datech')
plt.tight_layout()
plt.show()


**Interpretace:** U `BuildingArea` chybí 47,50 % hodnot, u `YearBuilt` 39,58 %,
u `CouncilArea` 10,08 % a u `Car` 0,46 %. Největší nejistota tedy souvisí s plochou a stářím.
Pouhé odstranění všech neúplných řádků by výrazně zmenšilo data a mohlo změnit jejich složení.
Skupinová imputace bude praktickým odhadem, nikoli rekonstrukcí skutečných hodnot.


### 1.5. Rozdělení dat a odlehlé hodnoty (Initial Exploration)
Nejprve prohlédneme původní hodnoty. Dlouhé pravé chvosty mohou znamenat skutečné velké objekty,
chyby zápisu nebo jinou definici plochy. Samotná odlehlost není důkaz chyby.


In [ ]:
# 1. Popisná statistika spojitých příznaků
continuous = ['Price', 'Distance', 'Landsize', 'BuildingArea']
display(df_raw[continuous + ['YearBuilt']].describe(percentiles=[.01, .5, .99]).round(2))

# 2. Rozdělení původních hodnot
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
units = ['Cena (AUD)', 'Vzdálenost (km)', 'Pozemek (m²)', 'Budova (m²)']
for ax, col, unit in zip(axes.flat, continuous, units):
    sns.histplot(df_raw[col].dropna(), bins=50, ax=ax, color='#337d9e')
    ax.set(xlabel=unit, ylabel='Počet záznamů', title=col)
plt.tight_layout()
plt.show()

# 3. Kontrola nul a největších ploch
print('Nulová plocha budovy:', df_raw['BuildingArea'].eq(0).sum())
print('Nulová plocha pozemku:', df_raw['Landsize'].eq(0).sum())
display(df_raw.nlargest(5, 'BuildingArea')[['Type', 'Rooms', 'BuildingArea', 'Landsize']])


**Interpretace:** Medián ceny je 903 000 AUD, maximum 9 000 000 AUD. U ploch jsou
rozdíly ještě výraznější: medián budovy 126 m² a maximum 44 515 m²; pozemek má maximum 433 014 m².
Extrémy stlačují většinu hodnot na levý okraj histogramu. Pro clustering proto použijeme
logaritmickou transformaci ploch a omezení jejich krajních hodnot; původní hodnoty zachováme pro popis.

Nulovou plochu budovy (17 případů) nemůžeme použít jako skutečnou obytnou plochu.
Nulový pozemek (1 939 případů) ale automaticky nevymažeme: například u bytů nemusí být samostatný
pozemek uveden. Jeho význam není v samotném CSV jednoznačný.


---

## 2. Čištění dat (Data Cleaning & Imputation)

### 2.1. Kontrola dat a neplatných hodnot

Datum prodeje převedeme z textu na `datetime` ve formátu **DD/MM/YYYY**. Následně označíme hodnoty, které nelze přímo použít:

- **BuildingArea:** Nekladnou plochu označíme jako chybějící.
- **YearBuilt:** Rok pod 1800 nebo po roce prodeje označíme jako nejistý pro výpočet stáří. Hranice 1800 je pracovní pravidlo; pozdější rok může znamenat i prodej před dokončením.
- **CouncilArea:** Chybějící údaj nahradíme kategorií `Unknown`. Tento sloupec do modelu nevstupuje.

### 2.2. Doplnění chybějících hodnot

**Zvolená strategie (Justified Strategy):** Použijeme skupinové mediány, které respektují typ nemovitosti, počet pokojů nebo lokalitu.

| Proměnná | Hlavní strategie | Záložní strategie |
|---|---|---|
| `BuildingArea` | Medián podle typu a počtu pokojů | Medián podle typu, poté celkový medián |
| `YearBuilt` | Medián podle předměstí a typu | Medián podle typu, poté celkový medián |
| `Car` | Medián podle typu | Celkový medián |

**Zdůvodnění:** Medián je méně citlivý na extrémy než průměr. Záložní strategie řeší skupiny bez známých hodnot. Doplněné údaje jsou však odhady a mohou vytvářet umělé podobnosti mezi objekty; jejich podíl proto sledujeme.


In [ ]:
# 1. Kopie dat a převod data prodeje
df = df_raw.drop_duplicates().copy()
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y', errors='coerce')
print('Nezpracovaná data prodeje:', df['Date'].isna().sum())

# 2. Označení neplatné plochy a nejistého roku
bad_area = df['BuildingArea'].notna() & df['BuildingArea'].le(0)
bad_year = df['YearBuilt'].notna() & (
    df['YearBuilt'].lt(1800) | df['YearBuilt'].gt(df['Date'].dt.year)
)
print('Neplatné/nekladné plochy:', bad_area.sum())
print('Nejisté roky výstavby:', bad_year.sum())
df.loc[bad_area, 'BuildingArea'] = np.nan
df.loc[bad_year, 'YearBuilt'] = np.nan

# 3. Uchování informace o původně chybějících údajích
df['BuildingAreaMissing'] = df['BuildingArea'].isna()
df['YearBuiltMissing'] = df['YearBuilt'].isna()

# 4. Skupinová imputace s náhradními úrovněmi
df['BuildingArea'] = df['BuildingArea'].fillna(
    df.groupby(['Type', 'Rooms'])['BuildingArea'].transform('median')
).fillna(df.groupby('Type')['BuildingArea'].transform('median')).fillna(df['BuildingArea'].median())
df['YearBuilt'] = df['YearBuilt'].fillna(
    df.groupby(['Suburb', 'Type'])['YearBuilt'].transform('median')
).fillna(df.groupby('Type')['YearBuilt'].transform('median')).fillna(df['YearBuilt'].median())
df['Car'] = df['Car'].fillna(df.groupby('Type')['Car'].transform('median')).fillna(df['Car'].median())
df['CouncilArea'] = df['CouncilArea'].fillna('Unknown')

# 5. Výsledná kontrola
print('Počet řádků po čištění:', len(df))
print('Zbývající chybějící hodnoty:', df.isna().sum().sum())


---

## 3. Tvorba nových příznaků (Feature Engineering)

Vytvoříme tři proměnné, které rozšiřují popis stáří, prostornosti a časového pokrytí dat.

| Příznak | Výpočet | Důvod |
|---|---|---|
| `BuildingAge` | Rok prodeje − rok výstavby | Stáří v okamžiku prodeje, použité v clusteringu. |
| `AreaPerRoom` | Plocha budovy / počet pokojů | Hrubé srovnání prostornosti; nezaměňujeme za skutečnou velikost konkrétního pokoje. |
| `SaleMonth` | Měsíc z data prodeje | Popis časového pokrytí dat; vhodný k odhalení neúplných období. |

`AreaPerRoom` nebude vstupem clusteringu současně s plochou a počtem pokojů,
abychom stejnou informaci nezapočítali ještě jednou. `SaleMonth` nepoužijeme jako běžné číslo
ve vzdálenosti: prosinec a leden jsou sousední měsíce, přesto mají hodnoty 12 a 1.


In [ ]:
# 1. Stáří budovy v okamžiku prodeje
df['BuildingAge'] = df['Date'].dt.year - df['YearBuilt']

# 2. Plocha připadající na jeden pokoj
df['AreaPerRoom'] = df['BuildingArea'] / df['Rooms'].replace(0, np.nan)

# 3. Měsíc prodeje
df['SaleMonth'] = df['Date'].dt.month
display(df[['BuildingAge', 'AreaPerRoom', 'SaleMonth']].describe().round(2))
print('Období:', df['Date'].min().date(), 'až', df['Date'].max().date())
print('Záporné stáří po imputaci:', df['BuildingAge'].lt(0).sum())


---

## 4. Průzkumná analýza dat (Deep Exploratory Data Analysis)

V této části zkoumáme rozdíly mezi typy nemovitostí, vztahy s cenou a korelace. Každý graf doplníme interpretací.
### 4.1. Typ nemovitosti a rozdělení cen
Kategorie `h` označuje domy, `u` jednotky/byty a `t` townhouses.
Porovnáváme zastoupení typů i jejich cenová rozdělení, nikoli pouze průměrnou cenu.


In [ ]:
# 1. Počty objektů a mediány cen podle typu
type_summary = df.groupby('Type').agg(pocet=('Price', 'size'), median_ceny=('Price', 'median'))
display(type_summary)

# 2. Porovnání zastoupení a cenových rozdělení
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(data=df, x='Type', order=['h', 'u', 't'], color='#337d9e', ax=axes[0])
axes[0].set(title='Zastoupení typů', xlabel='Typ', ylabel='Počet')
sns.boxplot(data=df, x='Type', y='Price', order=['h', 'u', 't'], color='#88bfd1', ax=axes[1])
axes[1].set_yscale('log')
axes[1].set(title='Ceny podle typu (logaritmická osa)', xlabel='Typ', ylabel='Cena (AUD, log škála)')
plt.tight_layout()
plt.show()


**Interpretace:** Domy tvoří 69.6% záznamů.
Mediány cen jsou 1,080,000 AUD pro domy,
560,000 AUD pro byty/jednotky a 846,750 AUD pro townhouses.
Rozdělení se překrývají: samotný typ cenu neurčuje. Logaritmická osa umožňuje vidět i levnější objekty.
Z těchto dat nelze usuzovat, pro koho je bydlení dostupné — neobsahují příjmy kupujících.

### 4.2. Vztah vzdálenosti a plochy k ceně (Bivariate Analysis)

Porovnáme cenu se vzdáleností od centra a se známou plochou budovy. Sílu monotónního vztahu doplníme Spearmanovou korelací.


In [ ]:
# 1. Pouze skutečně uvedené kladné plochy
observed_area = df.loc[~df['BuildingAreaMissing']]

# 2. Spearmanovy korelace
distance_corr = df['Distance'].corr(df['Price'], method='spearman')
area_corr = observed_area['BuildingArea'].corr(observed_area['Price'], method='spearman')
print(f'Spearman: vzdálenost a cena = {distance_corr:.3f}')
print(f'Spearman: známá plocha a cena = {area_corr:.3f}')

# 3. Bodové grafy vztahů s cenou
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.scatterplot(data=df, x='Distance', y='Price', hue='Type', alpha=.25, s=12, ax=axes[0])
axes[0].set_yscale('log')
axes[0].set(xlabel='Vzdálenost od centra (km)', ylabel='Cena (AUD, log škála)', title='Vzdálenost a cena')
sns.scatterplot(data=observed_area, x='BuildingArea', y='Price', hue='Type', alpha=.3, s=12, ax=axes[1])
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set(xlabel='Známá plocha budovy (m², log škála)', ylabel='Cena (AUD, log škála)', title='Pozorovaná plocha a cena')
plt.tight_layout()
plt.show()


**Interpretace:** Spearmanova korelace vzdálenosti s cenou je -0.130,
zatímco u známé plochy budovy a ceny je 0.636.
Jde o celkové vztahy napříč různými typy a lokalitami, nikoli o odhad kauzálního vlivu.
V grafu plochy používáme pouze skutečně uvedené kladné plochy, nikoli imputované hodnoty.
Rozptyl cen ukazuje, že ani jeden z těchto příznaků sám nevysvětluje celý trh.
Graf ani korelace nedokazují exponenciální vztah.

### 4.3. Korelační matice (Multivariate Analysis)

Prozkoumáme vztahy číselných příznaků a identifikujeme podobné informace, které by mohly při shlukování dostat příliš velkou váhu.


In [ ]:
# 1. Výběr číselných proměnných
eda_cols = ['Price', 'Rooms', 'Bedroom2', 'Bathroom', 'Car', 'Distance', 'Landsize', 'BuildingArea', 'BuildingAge']
correlations = df[eda_cols].corr(method='spearman')

# 2. Korelační matice s hodnotami
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(correlations, annot=True, fmt='.2f', cmap='vlag', vmin=-1, vmax=1, center=0, ax=ax)
ax.set_title('Spearmanovy korelace po imputaci')
plt.tight_layout()
plt.show()


**Interpretace:** Korelace `Rooms` a `Bedroom2` je 0.96.
Tyto sloupce nejsou ve všech řádcích totožné, ale představují velmi podobnou informaci.
Do clusteringu zahrneme pouze `Rooms`. Nezahrneme současně rok výstavby a odvozené stáří.
Korelace v této matici již ovlivňuje imputace: například doplnění plochy podle počtu pokojů
může jejich vztah uměle zesílit. Korelace neznamená příčinnou souvislost.

### 4.4. Časové pokrytí a plocha na pokoj

Zkontrolujeme zastoupení měsíců a rozdělení odvozeného příznaku `AreaPerRoom`.


In [ ]:
# 1. Časové pokrytí a plocha na pokoj
monthly = df.groupby(df['Date'].dt.to_period('M')).size()
area_per_room = df.loc[~df['BuildingAreaMissing'], 'AreaPerRoom']

# 2. Omezení horního 1 % pouze pro čitelnost histogramu
plot_limit = area_per_room.quantile(.99)

# 3. Vizualizace obou odvozených pohledů
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
monthly.plot.bar(ax=axes[0], color='#337d9e')
axes[0].set(title='Časové pokrytí záznamů', xlabel='Rok a měsíc prodeje', ylabel='Počet záznamů')
sns.histplot(area_per_room[area_per_room <= plot_limit], bins=35, ax=axes[1], color='#d58b4c')
axes[1].set(title='Plocha na pokoj: známé plochy, do 99. percentilu', xlabel='Plocha / počet pokojů (m²)', ylabel='Počet záznamů')
plt.tight_layout()
plt.show()
print('Medián známé plochy na pokoj:', round(area_per_room.median(), 2))


**Interpretace:** Záznamy pokrývají období od 28. 1. 2016 do 23. 9. 2017.
Měsíce nejsou zastoupeny stejně a poslední rok není úplný. Rozdíly v počtech proto nelze
automaticky označit za sezónnost celého trhu. Medián plochy na pokoj mezi známými plochami je
44.3 m². Jde o podíl celkové plochy budovy a počtu pokojů, ne o změřenou
plochu jednoho pokoje. Horní 1 % hodnot je vynecháno pouze z tohoto histogramu pro čitelnost.


---

## 5. Příprava dat pro shlukování (Feature Selection & Scaling)

### 5.1. Výběr příznaků
Použijeme **Rooms, Bathroom, Car, Distance, BuildingArea, Landsize, BuildingAge a Type**.
Cena slouží jen k následnému popisu. Adresu, prodejce, datum a správní názvy vynecháváme,
protože cílem jsou fyzické charakteristiky a vzdálenost od centra, nikoli skupiny podle prodejce.
Souřadnice by posílily geografii, ale pro tento jednoduchý model zastupuje polohu vzdálenost od centra.
To omezuje schopnost rozlišit různě drahé čtvrti ve stejné vzdálenosti.

### 5.2. Ošetření extrémů, transformace a standardizace
U obou ploch omezíme hodnoty na 1. a 99. percentil (**winsorizace**) pouze v modelovací kopii.
Jde o robustní pracovní volbu, ne o tvrzení, že všechna krajní pozorování jsou chybná.
Následuje `log1p(x) = log(1 + x)`, který zmenší pravostrannou šikmost a umožní zachovat nulové pozemky.

Číselné vstupy standardizujeme: $z = (x-\mu)/\sigma$.
`Type` převedeme na tři indikátory 0/1 pomocí one-hot encodingu; ty dále nestandardizujeme,
aby vzácnější typ nedostal automaticky vyšší váhu. Volba vah přesto ovlivňuje výsledky.


In [ ]:
# 1. Výběr modelovacích příznaků
numeric = ['Rooms', 'Bathroom', 'Car', 'Distance', 'BuildingArea', 'Landsize', 'BuildingAge']
model_data = df[numeric].copy()

# 2. Omezení extrémů ploch a logaritmická transformace
cap_rows = []
for col in ['BuildingArea', 'Landsize']:
    low, high = model_data[col].quantile([.01, .99])
    changed = ((model_data[col] < low) | (model_data[col] > high)).sum()
    cap_rows.append({'příznak': col, 'dolní mez': low, 'horní mez': high, 'upravené hodnoty': changed})
    model_data[col] = np.log1p(model_data[col].clip(lower=low, upper=high))
display(pd.DataFrame(cap_rows).round(2))

# 3. Standardizace číselných proměnných
scaler = StandardScaler()
X_num = pd.DataFrame(scaler.fit_transform(model_data), columns=numeric, index=df.index)

# 4. One-hot encoding typu a spojení vstupů
X_cat = pd.get_dummies(df['Type'], prefix='Type', dtype=float)
X_all = pd.concat([X_num, X_cat], axis=1)
print('Rozměry modelovací matice:', X_all.shape)
assert np.isfinite(X_all.to_numpy()).all(), 'Modelovací data obsahují NaN nebo nekonečno.'


### 5.3. Výběr společného vzorku
EDA používá všechna data. Oba algoritmy porovnáme na **stejném náhodném vzorku 5 000 objektů**.
Hierarchická metoda potřebuje přibližně kvadratickou paměť, proto je tento rozsah praktičtější
pro běžný notebook. Vzorek má pevný seed a není zaručeně reprezentativní; níže kontrolujeme
alespoň zastoupení typů. Výsledné profily platí pro tento vzorek, nikoli pro všechny nemovitosti v Melbourne.

Škálování zde využívá celou zkoumanou tabulku. Jde o popisnou analýzu bez odděleného testovacího souboru;
v budoucím predikčním projektu by se imputace a škálování musely učit pouze na trénovacích datech.


In [ ]:
# 1. Stejný náhodný vzorek pro oba algoritmy
sample_index = X_all.sample(n=min(5000, len(X_all)), random_state=SEED).index
X = X_all.loc[sample_index].copy()
sample = df.loc[sample_index].copy()

# 2. Kontrola zastoupení typů
type_comparison = pd.concat([
    df['Type'].value_counts(normalize=True).rename('celá data'),
    sample['Type'].value_counts(normalize=True).rename('vzorek')
], axis=1).mul(100).round(2)
display(type_comparison)
print('Velikost vzorku:', len(sample))


---

## 6. Shluková analýza (Cluster Analysis)

### 6.1. Použité algoritmy a hodnoticí metriky
**K-Means** minimalizuje součet čtverců vzdáleností bodů od centroidů (inertia).
Vyšší počet skupin tento součet automaticky zmenšuje, proto samotné minimum nestačí.
Použijeme i **silhouette**: $s=(b-a)/\max(a,b)$, kde $a$ je průměrná vzdálenost uvnitř vlastní skupiny
a $b$ nejmenší průměrná vzdálenost k jiné skupině. Vyšší hodnota znamená lepší oddělení v použitém prostoru.

**Hierarchické shlukování s Wardovou metodou** postupně spojuje skupiny s nejmenším nárůstem
vnitroskupinového součtu čtverců. Obě metody používají euklidovskou geometrii, a jejich shoda tedy
není nezávislým důkazem existence přirozených segmentů.

### 6.2. Hledání vhodného počtu shluků

Pro obě metody vyzkoušíme $k=2,\ldots,8$. Silhouette počítáme na stejných 2 000 vybraných objektech,
abychom snížili výpočetní nároky. Jde o odhad skóre. Pro každou metodu vybereme nejvyšší zjištěné skóre
a zároveň zkontrolujeme velikosti shluků a interpretovatelnost.


In [ ]:
# 1. Rozsah k a společný hodnoticí vzorek
k_values = range(2, 9)
eval_positions = np.random.default_rng(SEED).choice(len(X), size=min(2000, len(X)), replace=False)
X_eval = X.iloc[eval_positions]
results = []
km_labels = {}
ward_labels = {}

# 2. Výpočet hierarchie a porovnání metod pro jednotlivá k
ward_tree = linkage(X.to_numpy(), method='ward')
for k in k_values:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels_km = km.fit_predict(X)
    labels_ward = cut_tree(ward_tree, n_clusters=k).ravel()
    km_labels[k] = labels_km
    ward_labels[k] = labels_ward
    for method, labels, inertia in [
        ('K-Means', labels_km, km.inertia_),
        ('Ward', labels_ward, np.nan)
    ]:
        sizes = pd.Series(labels).value_counts()
        score = silhouette_score(X_eval, labels[eval_positions])
        results.append({'metoda': method, 'k': k, 'silhouette': score,
                        'inertia': inertia, 'nejmenší shluk': sizes.min(), 'největší shluk': sizes.max()})

# 3. Tabulka a grafy hodnoticích metrik
metrics = pd.DataFrame(results)
display(metrics.round(3))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.lineplot(data=metrics.query("metoda == 'K-Means'"), x='k', y='inertia', marker='o', ax=axes[0])
axes[0].set(title='K-Means: inertia', ylabel='Součet čtverců vzdáleností')
sns.lineplot(data=metrics, x='k', y='silhouette', hue='metoda', marker='o', ax=axes[1])
axes[1].set(title='Silhouette na stejném hodnoticím vzorku', ylabel='Silhouette')
plt.tight_layout()
plt.show()

# 4. Výběr k podle nejvyššího testovaného silhouette
best_k_km = int(metrics.query("metoda == 'K-Means'").sort_values('silhouette', ascending=False).iloc[0]['k'])
best_k_ward = int(metrics.query("metoda == 'Ward'").sort_values('silhouette', ascending=False).iloc[0]['k'])
print('Vybrané k — K-Means:', best_k_km, '| Ward:', best_k_ward)


**Vyhodnocení metrik (Finding & Decision):**

| Algoritmus | Vybraný počet shluků | Nejvyšší testované silhouette |
|---|---:|---:|
| K-Means | **2** | **0,249** |
| Ward | **3** | **0,225** |

Inertia s rostoucím počtem skupin klesá, ale graf neukazuje jednoznačný loket. Silhouette kolem 0,22–0,25 naznačuje omezené oddělení skupin.

**Rozhodnutí:** Pro K-Means zvolíme dvě skupiny, pro Ward tři. U K-Means je výsledek pro k=8 (přibližně 0,245) blízko k=2, proto dvě skupiny považujeme za jednoduchou výchozí segmentaci. Nejde o důkaz jediného správného počtu segmentů. Výběr závisí na příznacích, měřítku, vzorku a testovaném rozsahu k=2–8.

### 6.3. Finální modely a hierarchické shlukování

Porovnáme přiřazení objektů pomocí **Adjusted Rand Index (ARI)** a zobrazíme Wardův dendrogram. Výšku řezu odvodíme z vypočteného stromu pro zvolené k.


In [ ]:
# 1. Finální Wardův model a štítky obou metod
ward_final = AgglomerativeClustering(n_clusters=best_k_ward, linkage='ward')
final_ward_labels = ward_final.fit_predict(X)
sample['Cluster_KMeans'] = km_labels[best_k_km]
sample['Cluster_Ward'] = final_ward_labels

# 2. Shoda rozdělení a kontingenční tabulka
ari = adjusted_rand_score(sample['Cluster_KMeans'], sample['Cluster_Ward'])
print(f'Shoda K-Means vs. Ward (ARI): {ari:.3f}')
print('Kontrola řezu stromu vs. finální Ward:', round(adjusted_rand_score(ward_labels[best_k_ward], final_ward_labels), 3))
display(pd.crosstab(sample['Cluster_KMeans'], sample['Cluster_Ward']))

# 3. Dendrogram s řezem pro zvolené k
fig, ax = plt.subplots(figsize=(11, 4))
dendrogram(ward_tree, truncate_mode='lastp', p=15, ax=ax)
cut_height = (ward_tree[-best_k_ward, 2] + ward_tree[-best_k_ward + 1, 2]) / 2
ax.axhline(cut_height, color='crimson', linestyle='--', label=f'Řez pro k={best_k_ward}')
ax.set(title='Ward: zkrácený dendrogram', xlabel='Koncové skupiny (v závorkách počet objektů)', ylabel='Výška sloučení podle Wardovy metody')
ax.legend()
plt.tight_layout()
plt.show()


**Interpretace (Finding & Evidence):**

Dendrogram zobrazuje posloupnost slučování. Červená čára vyznačuje řez pro **tři skupiny**; sama o sobě nepředstavuje další nezávislý test správnosti počtu shluků.

**Shoda metod:** ARI dosahuje **0,240**, tedy pouze částečné shody. Algoritmy rozdělují objekty odlišně, také kvůli různému počtu skupin (2 a 3). Kontingenční tabulka ukazuje jejich překryv.

**Význam metriky:** ARI = 1 znamená totožné rozdělení až na názvy štítků; hodnota kolem 0 odpovídá náhodné shodě. Metrika měří shodu rozdělení, nikoli jejich pravdivost. Čísla shluků nejsou pořadím kvality.


### 6.4. Profily shluků v původních jednotkách
Profily počítáme v původních jednotkách. U ploch a stáří uvádíme také podíl imputovaných hodnot,
aby bylo zřejmé, nakolik jsou mediány založeny na odhadech. Ceny nebyly vstupem modelu.


In [ ]:
# 1. Profilové charakteristiky v původních jednotkách
profile_columns = ['Rooms', 'Bathroom', 'Car', 'Distance', 'BuildingArea', 'Landsize', 'BuildingAge', 'Price']
profiles = {}

# 2. Mediány, počty a podíly imputace pro obě metody
for method, label_col in [('K-Means', 'Cluster_KMeans'), ('Ward', 'Cluster_Ward')]:
    profile = sample.groupby(label_col)[profile_columns].median()
    profile.insert(0, 'počet', sample.groupby(label_col).size())
    profile['plocha doplněna (%)'] = sample.groupby(label_col)['BuildingAreaMissing'].mean().mul(100)
    profile['rok doplněn (%)'] = sample.groupby(label_col)['YearBuiltMissing'].mean().mul(100)
    profiles[method] = profile
    print(method, '— mediány a velikosti:')
    display(profile.round(2))
    print('Podíly typů uvnitř shluků (%):')
    display(pd.crosstab(sample[label_col], sample['Type'], normalize='index').mul(100).round(1))


### 6.5. Interpretace K-Means (Real-world Interpretation)

**Shluk 0 — Větší a dražší objekty dále od centra**

- **Počet:** 3 081 objektů.
- **Charakteristika:** Medián 3 pokojů, plocha 142 m², vzdálenost 11,2 km a stáří 56 let.
- **Cena:** Medián 1 100 000 AUD.
- **Interpretace:** Skupina zahrnuje relativně větší objekty ve větší vzdálenosti od centra. Ve srovnání s druhou skupinou mají také vyšší mediánovou cenu.

**Shluk 1 — Menší a levnější objekty blíže centru**

- **Počet:** 1 919 objektů.
- **Charakteristika:** Medián 2 pokojů, plocha 86 m², vzdálenost 6,4 km a stáří 47 let.
- **Cena:** Medián 691 000 AUD.
- **Interpretace:** Objekty jsou menší a blíže centru. Nižší mediánová cena ukazuje, že kratší vzdálenost sama o sobě nezaručuje vyšší cenu.

**Omezení interpretace:** Plocha byla doplněna u 47,8 % a 47,7 % objektů. Profily popisují tento vzorek; neumožňují určit příjem, věk ani rodinnou situaci kupujících.


### 6.6. Interpretace hierarchického shlukování (Ward)

**Shluk 0 — Starší objekty střední velikosti**

- **Počet:** 2 392 objektů.
- **Charakteristika:** Medián 3 pokojů, 117 m², vzdálenost 8,4 km a stáří 66 let.
- **Cena:** Medián 919 000 AUD.
- **Interpretace:** Nejstarší ze tří skupin s prostřední mediánovou plochou. Plocha byla doplněna u 52,5 % objektů.

**Shluk 1 — Větší objekty dále od centra**

- **Počet:** 1 907 objektů.
- **Charakteristika:** Medián 4 pokojů, 181 m², vzdálenost 12,4 km a stáří 43 let.
- **Cena:** Medián 1 080 000 AUD.
- **Interpretace:** Skupina s největší mediánovou plochou, největší vzdáleností a nejvyšší mediánovou cenou. Plocha byla doplněna u 43,5 % objektů.

**Shluk 2 — Menší objekty blíže centru**

- **Počet:** 701 objektů.
- **Charakteristika:** Medián 2 pokojů, 76 m², vzdálenost 5,9 km a stáří 46 let.
- **Cena:** Medián 557 000 AUD.
- **Interpretace:** Nejmenší skupina má nejnižší mediánovou cenu a nejmenší vzdálenost od centra. Plocha byla doplněna u 43,4 % objektů.

### 6.7. Vizualizace shluků pomocí PCA

Převedeme modelovací příznaky na dvě hlavní komponenty pro zobrazení obou rozdělení. Samotný clustering byl proveden v původním modelovacím prostoru.


In [ ]:
# 1. Dvourozměrná projekce pro vizualizaci
pca = PCA(n_components=2)
projection = pca.fit_transform(X)
explained = pca.explained_variance_ratio_.sum()

# 2. Zobrazení obou rozdělení ve stejné projekci
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, label_col, title in zip(axes, ['Cluster_KMeans', 'Cluster_Ward'], ['K-Means', 'Ward']):
    sns.scatterplot(x=projection[:, 0], y=projection[:, 1], hue=sample[label_col].astype(str).to_numpy(),
                    palette='tab10', s=13, alpha=.45, ax=ax)
    ax.set(xlabel='První hlavní komponenta', ylabel='Druhá hlavní komponenta', title=title)
    ax.legend(title='Shluk', fontsize=8)
plt.tight_layout()
plt.show()
print(f'Podíl rozptylu zachycený dvěma komponentami: {explained:.1%}')


**Interpretace:** PCA zde slouží pouze k zobrazení; clustering proběhl ve všech vybraných vstupních
příznacích. Dvě komponenty zachycují **61.1%** rozptylu, takže část informace nevidíme.
Překrytí bodů v obrázku nemusí znamenat stejné překrytí v původním prostoru.
Barvy znázorňují přiřazené shluky, nikoli známé správné třídy.


### 6.8. Kontrola citlivosti a stability (Sensitivity Analysis)
**A. Vynechání plochy budovy**

Vzhledem k rozsáhlé imputaci porovnáme K-Means se stejným k a na stejných objektech,
ale bez `BuildingArea`. Stejné škálování ostatních sloupců zachováme.
Tato kontrola ukazuje vliv příznaku jako celku; sama neodděluje vliv imputace od vlivu skutečné plochy.
Navíc zůstává imputované stáří, takže tím nejistota předzpracování nemizí.

**B. Změna náhodné inicializace**

Zopakujeme K-Means pro tři různé náhodné inicializace při pevném k.
To ověřuje stabilitu optimalizace na stejných datech, nikoli stabilitu při změně vzorku nebo období.


In [ ]:
# 1. Opakování K-Means bez plochy budovy
X_without_area = X.drop(columns='BuildingArea')
labels_without_area = KMeans(n_clusters=best_k_km, n_init=10, random_state=SEED).fit_predict(X_without_area)
area_ari = adjusted_rand_score(sample['Cluster_KMeans'], labels_without_area)
print(f'ARI s plochou vs. bez plochy: {area_ari:.3f}')

# 2. Porovnání různých náhodných inicializací
seed_results = []
for seed in [7, 21, 84]:
    labels = KMeans(n_clusters=best_k_km, n_init=10, random_state=seed).fit_predict(X)
    seed_results.append({'seed': seed, 'ARI proti seed 42': adjusted_rand_score(sample['Cluster_KMeans'], labels)})
stability = pd.DataFrame(seed_results)
display(stability.round(3))


**Interpretace (Robustness Check):**

- **Bez plochy budovy:** ARI = **0,784**. Velká část rozdělení zůstává podobná, ale některá přiřazení se mění. Vliv plochy nelze zanedbat.
- **Jiné inicializace:** ARI = **0,978–0,979** vůči seed 42. K-Means je na stejných datech při těchto inicializacích poměrně stabilní.

Kontrola bez plochy měří vliv celého příznaku, nikoli jen imputace. Stabilita vůči inicializaci rovněž neznamená stabilitu při změně vzorku, období nebo způsobu předzpracování.


---

## 7. Závěr 1. části projektu (Method-Selection Memo)

Na základě EDA a shlukové analýzy jsme splnili hlavní cíle první části projektu:

1. **Příprava dat:** Zpracovali jsme **13 580 záznamů**, ošetřili neplatné hodnoty a doplnili chybějící údaje skupinovými mediány. Vytvořili jsme nové příznaky `BuildingAge`, `AreaPerRoom` a `SaleMonth`.

2. **Vyhodnocení algoritmů:** Na společném vzorku **5 000 nemovitostí** jsme porovnali K-Means a hierarchické shlukování s Wardovou metodou. Podle silhouette skóre jsme zvolili **2 shluky pro K-Means (0,249)** a **3 shluky pro Ward (0,225)**. Lepšího skóre dosáhl K-Means, přesto oddělení skupin není výrazné.

3. **Interpretace shluků:** K-Means rozlišil dvě hlavní skupiny: **větší a dražší objekty dále od centra** a **menší a levnější objekty blíže centru**. Cena nebyla vstupním příznakem; použili jsme ji až k následnému porovnání skupin.

Výsledky poskytují přehled o rozdílech mezi nemovitostmi ve zkoumaném vzorku. Při jejich interpretaci je nutné zohlednit vysoký podíl doplněných hodnot plochy a roku výstavby.
Provedená analýza nám pomohla lépe porozumět datům a poskytla základ pro další práci ve 2.části projektu.

---

## Příloha: Zdroje

**Data:** [Melbourne Housing Snapshot — Kaggle](https://www.kaggle.com/datasets/dansbecker/melbourne-housing-snapshot)  
**Zadání:** [Fundamentals of Machine Learning — semestrální projekt](https://homel.vsb.cz/~pla06/subject_fml.html)
